# 读取 test 目录四类数据

这个 notebook 用于读取 `test` 目录中的四类文件：

- `Z9002_*.npy`：雷达反射率数据
- `wind_*.npy` / `wind_*.csv` / `wind_*.parquet`：站点测风数据
- `Pre_1h_*.npy` / `Pre_1h_*.csv` / `pre_1h_*.parquet`：站点降水数据
- `radar_situation.npz`：雷达经纬度信息

默认只读取每类的第一个样例文件，避免一次性加载过多数据。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

TEST_DIR = Path('test')
assert TEST_DIR.exists(), f'目录不存在: {TEST_DIR.resolve()}'

TEST_DIR.resolve()

WindowsPath('D:/gaochu/相风/test')

## 1. 收集四类文件

In [2]:
radar_reflectivity_files = sorted(TEST_DIR.glob('Z9002_*.npy'))

wind_files = sorted(
    list(TEST_DIR.glob('wind_*.npy'))
    + list(TEST_DIR.glob('wind_*.csv'))
    + list(TEST_DIR.glob('wind_*.parquet'))
)

precipitation_files = sorted(
    list(TEST_DIR.glob('Pre_1h_*.npy'))
    + list(TEST_DIR.glob('Pre_1h_*.csv'))
    + list(TEST_DIR.glob('Pre_1h_*.parquet'))
    + list(TEST_DIR.glob('pre_1h_*.npy'))
    + list(TEST_DIR.glob('pre_1h_*.csv'))
    + list(TEST_DIR.glob('pre_1h_*.parquet'))
)

radar_situation_file = TEST_DIR / 'radar_situation.npz'

summary = pd.DataFrame(
    [
        {'类型': '雷达反射率', '匹配': 'Z9002_*.npy', '数量': len(radar_reflectivity_files)},
        {'类型': '站点测风数据', '匹配': 'wind_*.npy/csv/parquet', '数量': len(wind_files)},
        {'类型': '站点降水数据', '匹配': 'Pre_1h_/pre_1h_*.npy/csv/parquet', '数量': len(precipitation_files)},
        {'类型': '雷达经纬度', '匹配': 'radar_situation.npz', '数量': int(radar_situation_file.exists())},
    ]
)

summary

,类型,匹配,数量
0,雷达反射率,Z9002_*.npy,566
1,站点测风数据,wind_*.npy/csv/parquet,576
2,站点降水数据,Pre_1h_/pre_1h_*.npy/csv/parquet,1152
3,雷达经纬度,radar_situation.npz,1


## 2. 通用读取函数

In [3]:
def read_data_file(path: Path):
    """根据后缀读取 npy/csv/parquet/npz 文件。"""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == '.npy':
        return np.load(path, allow_pickle=False)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix == '.npz':
        return np.load(path, allow_pickle=False)

    raise ValueError(f'不支持的文件类型: {path}')


def describe_loaded(name: str, data):
    print(f'[{name}]')
    if isinstance(data, pd.DataFrame):
        print('类型: pandas.DataFrame')
        print('shape:', data.shape)
        display(data.head())
    elif isinstance(data, np.lib.npyio.NpzFile):
        print('类型: numpy.lib.npyio.NpzFile')
        print('keys:', list(data.keys()))
        for key in data.keys():
            value = data[key]
            print(f'  {key}: shape={value.shape}, dtype={value.dtype}')
    elif isinstance(data, np.ndarray):
        print('类型: numpy.ndarray')
        print('shape:', data.shape)
        print('dtype:', data.dtype)
        print('min/max:', np.nanmin(data), np.nanmax(data))
    else:
        print('类型:', type(data))
        print(data)

## 3. 读取雷达反射率 `Z9002_*.npy`

In [4]:
radar_reflectivity_sample = read_data_file(radar_reflectivity_files[0])
print('文件:', radar_reflectivity_files[0])
describe_loaded('雷达反射率样例', radar_reflectivity_sample)

文件: test\Z9002_20240526000744.npy
[雷达反射率样例]
类型: numpy.ndarray
shape: (461, 461)
dtype: float32
min/max: -2.2392294 52.908016


如需读取全部雷达反射率文件，可以使用下面的生成器逐个读取。

In [5]:
def iter_radar_reflectivity():
    for path in radar_reflectivity_files:
        yield path, np.load(path, allow_pickle=False)

# 示例：读取前三个
for path, arr in list(iter_radar_reflectivity())[:3]:
    print(path.name, arr.shape, arr.dtype)

Z9002_20240526000744.npy (461, 461) float32
Z9002_20240526001745.npy (461, 461) float32
Z9002_20240526002746.npy (461, 461) float32


## 4. 读取站点测风数据 `wind_*`

In [6]:
wind_sample = read_data_file(wind_files[0])
print('文件:', wind_files[0])
describe_loaded('站点测风数据样例', wind_sample)

文件: test\wind_2024052600.parquet
[站点测风数据样例]
类型: pandas.DataFrame
shape: (19555, 7)


,Lat,Lon,Alti,WIN_S_Avg_10mi,WIN_D_Avg_10mi,WIN_S_Inst_Max,WIN_S_INST_Max_OTime
0,39.4994,117.9733,3.4,3.7,299.0,5.6,2353.0
1,39.4989,121.9781,40.0,2.3,156.0,5.5,2334.0
2,39.4958,117.5186,5.0,3.6,301.0,6.9,2357.0
3,39.4947,115.7878,54.0,2.3,206.0,5.3,2330.0
4,39.4914,116.6939,18.6,1.6,241.0,5.2,2346.0


如需合并全部测风表格文件，可以使用下面代码。数据量较大时建议先筛选时间范围。

In [7]:
def read_many_table_files(paths):
    frames = []
    for path in paths:
        data = read_data_file(path)
        if isinstance(data, pd.DataFrame):
            data = data.copy()
            data['source_file'] = path.name
            frames.append(data)
        else:
            raise TypeError(f'{path} 读取结果不是 DataFrame，实际为 {type(data)}')
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# wind_all = read_many_table_files(wind_files)
# wind_all.shape

## 5. 读取站点降水数据 `Pre_1h_*` / `pre_1h_*`

In [8]:
precipitation_sample = read_data_file(precipitation_files[0])
print('文件:', precipitation_files[0])
describe_loaded('站点降水数据样例', precipitation_sample)

文件: test\pre_1h_2024052600.parquet
[站点降水数据样例]
类型: pandas.DataFrame
shape: (25667, 4)


,Lat,Lon,Alti,PRE_1h
0,39.4994,116.2656,37.0,0.0
1,39.4994,117.9733,3.4,0.0
2,39.4989,121.9781,40.0,0.2
3,39.4972,116.0764,27.0,0.0
4,39.4967,116.7331,16.8,0.0


In [9]:
# precipitation_all = read_many_table_files(precipitation_files)
# precipitation_all.shape

## 6. 读取雷达经纬度 `radar_situation.npz`

In [10]:
radar_situation = read_data_file(radar_situation_file)
print('文件:', radar_situation_file)
describe_loaded('雷达经纬度', radar_situation)

文件: test\radar_situation.npz
[雷达经纬度]
类型: numpy.lib.npyio.NpzFile
keys: ['lon', 'lat']
  lon: shape=(461, 461), dtype=float64
  lat: shape=(461, 461), dtype=float64


可以按 key 取出 `npz` 中的数组。

In [11]:
radar_situation_arrays = {key: radar_situation[key] for key in radar_situation.keys()}
radar_situation_arrays.keys()

dict_keys(['lon', 'lat'])